# 02. Procesos de Decisión de Markov y Ecuaciones de Bellman

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 75 minutos  
**Prerequisitos:** [01. Fundamentos de Agentes](01-fundamentos-agentes.ipynb)

## 🎯 Objetivos de Aprendizaje
Al finalizar este notebook, podrás:
- Comprender la formalización matemática de problemas de RL mediante MDPs
- Definir y calcular funciones de valor de estados y acciones
- Derivar y aplicar las ecuaciones de Bellman
- Implementar evaluación de políticas usando programación dinámica
- Entender los conceptos de política óptima y función de valor óptima

## 📚 Motivación

### El Problema de la Secuencialidad

Imagina que estás jugando ajedrez. Cada movimiento que haces no solo afecta tu posición actual, sino también todas las posiciones futuras posibles. ¿Cómo evalúas si un movimiento es "bueno"?

- Un movimiento puede parecer malo ahora pero llevar a una victoria eventualmente
- Un movimiento que captura una pieza puede ser una trampa
- Necesitas considerar las consecuencias a largo plazo

Este es el desafío fundamental del **aprendizaje por refuerzo**: tomar decisiones considerando no solo las recompensas inmediatas, sino también las recompensas futuras.

### ¿Por qué Necesitamos MDPs?

En el notebook anterior vimos agentes simples, pero no teníamos:
1. **Formalización matemática rigurosa** del problema
2. **Forma de evaluar políticas** objetivamente
3. **Teoría para encontrar políticas óptimas**
4. **Manejo de incertidumbre** en las transiciones

Los **Procesos de Decisión de Markov (MDPs)** proporcionan todo esto y más.

### Pregunta Guía
**¿Cómo podemos calcular matemáticamente qué tan valiosa es una acción considerando todas sus consecuencias futuras?**

Las ecuaciones de Bellman responden exactamente esta pregunta.

In [ ]:
# Importar librerías
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from typing import Tuple, List, Dict, Optional
import seaborn as sns

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
np.random.seed(42)

print("✅ Librerías importadas correctamente")

## 🎨 Intuición Visual

### Visualizando un MDP

Un MDP se puede visualizar como un grafo dirigido donde:
- **Nodos** = Estados
- **Aristas** = Transiciones (acciones)
- **Etiquetas** = Recompensas

Veamos un ejemplo simple: un robot que navega en una línea.

In [ ]:
def visualize_simple_mdp():
    """
    Visualiza un MDP simple: Robot en una línea de 5 posiciones.
    
    Estados: [0, 1, 2, 3, 4]
    Acciones: izquierda, derecha
    Objetivo: Llegar al estado 4
    """
    fig = go.Figure()
    
    # Estados
    states = [0, 1, 2, 3, 4]
    y_pos = [0] * len(states)
    
    # Dibujar estados
    colors = ['lightblue'] * 4 + ['lightgreen']  # Estado 4 es el objetivo
    fig.add_trace(go.Scatter(
        x=states,
        y=y_pos,
        mode='markers+text',
        marker=dict(size=50, color=colors, line=dict(width=2, color='black')),
        text=[f'S{i}' for i in states],
        textposition='middle center',
        textfont=dict(size=16, color='darkblue'),
        showlegend=False
    ))
    
    # Dibujar transiciones (flechas)
    # Acción: derecha
    for i in range(4):
        fig.add_annotation(
            x=i+1, y=0.3,
            ax=i, ay=0.3,
            xref='x', yref='y',
            axref='x', ayref='y',
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor='green'
        )
        # Etiqueta de recompensa
        reward = 10 if i == 3 else -1
        fig.add_annotation(
            x=(i + i+1)/2, y=0.5,
            text=f'r={reward}',
            showarrow=False,
            font=dict(size=10, color='green')
        )
    
    # Acción: izquierda
    for i in range(1, 5):
        fig.add_annotation(
            x=i-1, y=-0.3,
            ax=i, ay=-0.3,
            xref='x', yref='y',
            axref='x', ayref='y',
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor='red'
        )
        fig.add_annotation(
            x=(i + i-1)/2, y=-0.5,
            text=f'r=-1',
            showarrow=False,
            font=dict(size=10, color='red')
        )
    
    fig.update_layout(
        title='MDP Simple: Robot en Línea<br>Verde=Derecha, Rojo=Izquierda',
        xaxis=dict(showgrid=False, zeroline=False, range=[-0.5, 4.5]),
        yaxis=dict(showgrid=False, zeroline=False, range=[-1, 1]),
        height=400,
        template='plotly_white'
    )
    
    return fig

fig = visualize_simple_mdp()
fig.show()

print("\n📊 Componentes del MDP:")
print("  • Estados (S): {S0, S1, S2, S3, S4}")
print("  • Acciones (A): {izquierda, derecha}")
print("  • Recompensas (R): -1 por cada paso, +10 al llegar a S4")
print("  • Transiciones: Determinísticas (100% de probabilidad)")

### Función de Valor: ¿Qué tan bueno es un estado?

La **función de valor** $V(s)$ nos dice el retorno esperado si comenzamos en el estado $s$ y seguimos una política $\pi$.

In [ ]:
# Visualizar función de valor para diferentes políticas
def visualize_value_function():
    """
    Compara funciones de valor para dos políticas:
    1. Política Óptima: Siempre ir a la derecha
    2. Política Aleatoria: 50% izquierda, 50% derecha
    """
    states = ['S0', 'S1', 'S2', 'S3', 'S4']
    
    # Valores calculados manualmente (veremos cómo después)
    # Asumiendo γ=0.9 (factor de descuento)
    v_optimal = [6.56, 7.29, 8.10, 9.00, 10.0]  # Política óptima
    v_random = [2.0, 3.5, 5.0, 6.8, 10.0]        # Política aleatoria
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=states,
        y=v_optimal,
        name='Política Óptima',
        marker_color='lightgreen'
    ))
    
    fig.add_trace(go.Bar(
        x=states,
        y=v_random,
        name='Política Aleatoria',
        marker_color='lightcoral'
    ))
    
    fig.update_layout(
        title='Función de Valor V(s) para Diferentes Políticas',
        xaxis_title='Estado',
        yaxis_title='Valor V(s)',
        barmode='group',
        template='plotly_white',
        height=400
    )
    
    return fig

fig = visualize_value_function()
fig.show()

print("\n💡 Observación:")
print("  - Estados más cercanos al objetivo tienen mayor valor")
print("  - La política óptima tiene valores más altos en todos los estados")
print("  - El valor del objetivo (S4) es 10 para ambas políticas")
print("  - La función de valor captura consecuencias a largo plazo")

## 📐 Fundamentos Matemáticos

### Definición Formal de un MDP

Un **Proceso de Decisión de Markov** es una tupla $\langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$:

$$
\begin{align}
\mathcal{S} &: \text{conjunto finito de estados} \tag{1} \\
\mathcal{A} &: \text{conjunto finito de acciones} \tag{2} \\
P &: \text{función de transición de probabilidad} \tag{3} \\
R &: \text{función de recompensa} \tag{4} \\
\gamma &\in [0, 1]: \text{factor de descuento} \tag{5}
\end{align}
$$

#### Función de Transición

$$
\begin{align}
P(s'|s, a) &= P(s_{t+1} = s' \mid s_t = s, a_t = a) \tag{6} \\
\text{donde: } & \\
P(s'|s, a) &: \text{probabilidad de transicionar a } s' \text{ desde } s \text{ tomando acción } a
\end{align}
$$

**Propiedad de Markov**: El futuro es independiente del pasado dado el presente
$$P(s_{t+1} \mid s_t, s_{t-1}, \ldots, s_0) = P(s_{t+1} \mid s_t)$$

#### Factor de Descuento

El **factor de descuento** $\gamma$ controla la importancia de recompensas futuras:

$$
\begin{align}
G_t &= r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots \tag{7} \\
    &= \sum_{k=0}^{\infty} \gamma^k r_{t+k+1} \tag{8}
\end{align}
$$

- $\gamma = 0$: Solo importa la recompensa inmediata (miope)
- $\gamma \to 1$: Recompensas futuras importan tanto como las presentes (previsor)
- $\gamma < 1$ garantiza convergencia en sumas infinitas

### Funciones de Valor

#### Función de Valor de Estado

$$
\begin{align}
V^{\pi}(s) &= \mathbb{E}_{\pi}[G_t \mid s_t = s] \tag{9} \\
           &= \mathbb{E}_{\pi}\left[\sum_{k=0}^{\infty} \gamma^k r_{t+k+1} \mid s_t = s\right] \tag{10} \\
\text{donde: } & \\
V^{\pi}(s) &: \text{valor esperado de comenzar en } s \text{ y seguir política } \pi \\
\mathbb{E}_{\pi} &: \text{esperanza bajo política } \pi
\end{align}
$$

#### Función de Valor de Acción (Q-Function)

$$
\begin{align}
Q^{\pi}(s, a) &= \mathbb{E}_{\pi}[G_t \mid s_t = s, a_t = a] \tag{11} \\
              &= \mathbb{E}_{\pi}\left[\sum_{k=0}^{\infty} \gamma^k r_{t+k+1} \mid s_t = s, a_t = a\right] \tag{12} \\
\text{donde: } & \\
Q^{\pi}(s, a) &: \text{valor de tomar acción } a \text{ en estado } s \text{ y luego seguir } \pi
\end{align}
$$

#### Relación entre V y Q

$$
\begin{align}
V^{\pi}(s) &= \sum_{a \in \mathcal{A}} \pi(a|s) Q^{\pi}(s, a) \tag{13} \\
Q^{\pi}(s, a) &= \sum_{s' \in \mathcal{S}} P(s'|s,a)[R(s,a,s') + \gamma V^{\pi}(s')] \tag{14}
\end{align}
$$

### Ecuaciones de Bellman

Las **ecuaciones de Bellman** relacionan el valor de un estado con los valores de sus sucesores.

#### Ecuación de Bellman para $V^{\pi}$

$$
\begin{align}
V^{\pi}(s) &= \sum_{a \in \mathcal{A}} \pi(a|s) \sum_{s' \in \mathcal{S}} P(s'|s,a)[R(s,a,s') + \gamma V^{\pi}(s')] \tag{15}
\end{align}
$$

**Interpretación**: El valor de un estado es el promedio ponderado de:
- La recompensa inmediata esperada
- El valor (descontado) del siguiente estado

#### Ecuación de Bellman para $Q^{\pi}$

$$
\begin{align}
Q^{\pi}(s, a) &= \sum_{s' \in \mathcal{S}} P(s'|s,a)\left[R(s,a,s') + \gamma \sum_{a'} \pi(a'|s') Q^{\pi}(s', a')\right] \tag{16}
\end{align}
$$

### Optimalidad

#### Política Óptima

$$
\begin{align}
\pi^* &= \arg\max_{\pi} V^{\pi}(s), \quad \forall s \in \mathcal{S} \tag{17} \\
\text{donde: } & \\
\pi^* &: \text{política óptima (maximiza valor en todos los estados)}
\end{align}
$$

#### Ecuación de Optimalidad de Bellman

$$
\begin{align}
V^*(s) &= \max_{a \in \mathcal{A}} Q^*(s, a) \tag{18} \\
       &= \max_{a \in \mathcal{A}} \sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V^*(s')] \tag{19}
\end{align}
$$

$$
\begin{align}
Q^*(s, a) &= \sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma \max_{a'} Q^*(s', a')] \tag{20}
\end{align}
$$

**Diferencia clave**: En lugar de promediar sobre acciones según $\pi$, tomamos el **máximo**.

### Ejemplo Numérico

Consideremos el MDP de la línea con $\gamma = 0.9$:

**Estado S3** (un paso antes del objetivo):
- Acción derecha → S4: recompensa = +10
- Acción izquierda → S2: recompensa = -1

Usando la ecuación de Bellman para la política óptima (siempre derecha):

$$
\begin{align}
V^*(S_3) &= R + \gamma V^*(S_4) \\
         &= (-1) + 0.9 \times 10 \\
         &= -1 + 9 \\
         &= 8.0
\end{align}
$$

**Estado S2**:
$$
\begin{align}
V^*(S_2) &= (-1) + 0.9 \times V^*(S_3) \\
         &= -1 + 0.9 \times 8 \\
         &= -1 + 7.2 \\
         &= 6.2
\end{align}
$$

Y así sucesivamente hacia atrás.

> 💡 **Insight**: Las ecuaciones de Bellman nos permiten calcular valores de forma **recursiva** sin necesidad de simular trayectorias completas.

## 💻 Implementación Desde Cero

Implementaremos:
1. Un MDP en GridWorld
2. Evaluación de políticas (calcular $V^{\pi}$)
3. Iteración de valor (encontrar $V^*$)

In [ ]:
class GridWorldMDP:
    """
    MDP completo para GridWorld.
    
    Incluye funciones de transición probabilísticas y recompensas.
    """
    
    def __init__(self, size: int = 4, gamma: float = 0.9):
        self.size = size
        self.gamma = gamma  # Factor de descuento
        
        # Definir estados
        self.states = [(i, j) for i in range(size) for j in range(size)]
        self.n_states = len(self.states)
        
        # Definir acciones: 0=arriba, 1=derecha, 2=abajo, 3=izquierda
        self.actions = [0, 1, 2, 3]
        self.action_names = ['↑', '→', '↓', '←']
        self.n_actions = len(self.actions)
        
        # Mapeo de acciones a cambios de posición
        self.action_deltas = {
            0: (-1, 0),  # arriba
            1: (0, 1),   # derecha
            2: (1, 0),   # abajo
            3: (0, -1)   # izquierda
        }
        
        # Definir estados especiales
        self.goal_state = (size-1, size-1)  # Esquina inferior derecha
        self.obstacles = [(1, 1), (2, 2)] if size >= 3 else []
        
        # Estado inicial para visualización
        self.start_state = (0, 0)
    
    def get_next_state(self, state: Tuple[int, int], action: int) -> Tuple[int, int]:
        """
        Calcula el siguiente estado dada una acción (determinístico).
        
        Parameters:
        -----------
        state : tuple
            Estado actual (i, j)
        action : int
            Acción a tomar (0-3)
        
        Returns:
        --------
        next_state : tuple
            Siguiente estado
        """
        if state == self.goal_state:
            return state  # Estado terminal
        
        i, j = state
        di, dj = self.action_deltas[action]
        new_i, new_j = i + di, j + dj
        
        # Verificar límites
        if new_i < 0 or new_i >= self.size or new_j < 0 or new_j >= self.size:
            return state  # Se queda en el mismo lugar
        
        # Verificar obstáculos
        if (new_i, new_j) in self.obstacles:
            return state  # Se queda en el mismo lugar
        
        return (new_i, new_j)
    
    def get_reward(self, state: Tuple[int, int], action: int, next_state: Tuple[int, int]) -> float:
        """
        Calcula la recompensa de una transición.
        """
        if next_state == self.goal_state:
            return 10.0  # Recompensa por llegar al objetivo
        elif next_state == state:  # Chocó con obstáculo o límite
            return -1.0
        else:
            return -0.1  # Penalización pequeña por cada paso
    
    def get_transition_prob(self, state: Tuple[int, int], action: int, next_state: Tuple[int, int]) -> float:
        """
        Retorna P(next_state | state, action).
        
        Para este MDP simple, las transiciones son determinísticas.
        """
        expected_next = self.get_next_state(state, action)
        return 1.0 if next_state == expected_next else 0.0

# Crear MDP
mdp = GridWorldMDP(size=4, gamma=0.9)
print("🎲 MDP de GridWorld creado\n")
print(f"  Estados: {mdp.n_states}")
print(f"  Acciones: {mdp.n_actions} {mdp.action_names}")
print(f"  Factor de descuento γ: {mdp.gamma}")
print(f"  Objetivo: {mdp.goal_state}")
print(f"  Obstáculos: {mdp.obstacles}")

### Evaluación de Políticas (Policy Evaluation)

Algoritmo iterativo para calcular $V^{\pi}$ dada una política $\pi$.

In [ ]:
def policy_evaluation(mdp: GridWorldMDP, policy: np.ndarray, theta: float = 1e-6, max_iterations: int = 1000) -> np.ndarray:
    """
    Evalúa una política usando iteración de valor.
    
    Implementa la ecuación de Bellman iterativamente:
    V(s) ← Σ_a π(a|s) Σ_s' P(s'|s,a)[R(s,a,s') + γV(s')]
    
    Parameters:
    -----------
    mdp : GridWorldMDP
        Proceso de Decisión de Markov
    policy : np.ndarray
        Política π(a|s), shape (n_states, n_actions)
    theta : float
        Umbral de convergencia
    max_iterations : int
        Máximo número de iteraciones
    
    Returns:
    --------
    V : np.ndarray
        Función de valor V^π(s), shape (size, size)
    """
    # Inicializar función de valor a cero
    V = np.zeros((mdp.size, mdp.size))
    
    for iteration in range(max_iterations):
        delta = 0  # Seguimiento del cambio máximo
        V_old = V.copy()
        
        # Actualizar valor de cada estado
        for state in mdp.states:
            if state == mdp.goal_state:
                continue  # Estado terminal tiene valor 0
            
            i, j = state
            state_idx = i * mdp.size + j
            
            # Calcular nuevo valor usando ecuación de Bellman
            v = 0
            for action in mdp.actions:
                # Probabilidad de tomar esta acción bajo la política
                action_prob = policy[state_idx, action]
                
                # Calcular siguiente estado y recompensa
                next_state = mdp.get_next_state(state, action)
                reward = mdp.get_reward(state, action, next_state)
                
                # Bellman update
                next_i, next_j = next_state
                v += action_prob * (reward + mdp.gamma * V_old[next_i, next_j])
            
            V[i, j] = v
            delta = max(delta, abs(V[i, j] - V_old[i, j]))
        
        # Verificar convergencia
        if delta < theta:
            print(f"✅ Convergió en {iteration + 1} iteraciones")
            break
    else:
        print(f"⚠️ No convergió en {max_iterations} iteraciones")
    
    return V


# Crear política aleatoria uniforme
random_policy = np.ones((mdp.n_states, mdp.n_actions)) / mdp.n_actions

print("🔄 Evaluando política aleatoria uniforme...\n")
V_random = policy_evaluation(mdp, random_policy)

print("\nFunción de valor V^π:")
print(np.round(V_random, 2))
print("\n(Los valores más altos están cerca del objetivo)")

### Iteración de Valor (Value Iteration)

Encuentra la función de valor óptima $V^*$ directamente.

In [ ]:
def value_iteration(mdp: GridWorldMDP, theta: float = 1e-6, max_iterations: int = 1000) -> Tuple[np.ndarray, np.ndarray]:
    """
    Encuentra la política óptima usando iteración de valor.
    
    Implementa la ecuación de optimalidad de Bellman:
    V(s) ← max_a Σ_s' P(s'|s,a)[R(s,a,s') + γV(s')]
    
    Parameters:
    -----------
    mdp : GridWorldMDP
        Proceso de Decisión de Markov
    theta : float
        Umbral de convergencia
    max_iterations : int
        Máximo número de iteraciones
    
    Returns:
    --------
    V : np.ndarray
        Función de valor óptima V*(s)
    policy : np.ndarray
        Política óptima π*(a|s)
    """
    # Inicializar función de valor
    V = np.zeros((mdp.size, mdp.size))
    
    for iteration in range(max_iterations):
        delta = 0
        V_old = V.copy()
        
        # Actualizar cada estado
        for state in mdp.states:
            if state == mdp.goal_state:
                continue
            
            i, j = state
            
            # Calcular valor para cada acción
            action_values = []
            for action in mdp.actions:
                next_state = mdp.get_next_state(state, action)
                reward = mdp.get_reward(state, action, next_state)
                next_i, next_j = next_state
                
                # Valor de esta acción
                action_value = reward + mdp.gamma * V_old[next_i, next_j]
                action_values.append(action_value)
            
            # Tomar el máximo (ecuación de optimalidad de Bellman)
            V[i, j] = max(action_values)
            delta = max(delta, abs(V[i, j] - V_old[i, j]))
        
        if delta < theta:
            print(f"✅ Convergió en {iteration + 1} iteraciones")
            break
    else:
        print(f"⚠️ No convergió en {max_iterations} iteraciones")
    
    # Extraer política óptima
    policy = np.zeros((mdp.n_states, mdp.n_actions))
    
    for state in mdp.states:
        if state == mdp.goal_state:
            continue
        
        i, j = state
        state_idx = i * mdp.size + j
        
        # Encontrar la mejor acción
        action_values = []
        for action in mdp.actions:
            next_state = mdp.get_next_state(state, action)
            reward = mdp.get_reward(state, action, next_state)
            next_i, next_j = next_state
            action_value = reward + mdp.gamma * V[next_i, next_j]
            action_values.append(action_value)
        
        # Política determinística: probabilidad 1 a la mejor acción
        best_action = np.argmax(action_values)
        policy[state_idx, best_action] = 1.0
    
    return V, policy


print("🎯 Ejecutando iteración de valor para encontrar política óptima...\n")
V_optimal, optimal_policy = value_iteration(mdp)

print("\nFunción de valor óptima V*:")
print(np.round(V_optimal, 2))

print("\n📋 Política óptima (acciones):")
policy_grid = np.zeros((mdp.size, mdp.size), dtype=int)
for i in range(mdp.size):
    for j in range(mdp.size):
        state_idx = i * mdp.size + j
        policy_grid[i, j] = np.argmax(optimal_policy[state_idx])

# Visualizar con flechas
for i in range(mdp.size):
    row = ""
    for j in range(mdp.size):
        if (i, j) == mdp.goal_state:
            row += "G "
        elif (i, j) in mdp.obstacles:
            row += "X "
        else:
            action = policy_grid[i, j]
            row += mdp.action_names[action] + " "
    print(row)

### Visualización de Funciones de Valor

In [ ]:
def visualize_value_function_heatmap(V: np.ndarray, title: str = "Función de Valor"):
    """
    Visualiza la función de valor como un mapa de calor.
    """
    fig = go.Figure(data=go.Heatmap(
        z=V,
        colorscale='Viridis',
        text=np.round(V, 2),
        texttemplate='%{text}',
        textfont={"size": 14},
        colorbar=dict(title="Valor")
    ))
    
    fig.update_layout(
        title=title,
        xaxis=dict(title="Columna", dtick=1),
        yaxis=dict(title="Fila", dtick=1, autorange='reversed'),
        width=500,
        height=500
    )
    
    return fig

# Comparar funciones de valor
fig1 = visualize_value_function_heatmap(V_random, "V^π (Política Aleatoria)")
fig2 = visualize_value_function_heatmap(V_optimal, "V* (Política Óptima)")

fig1.show()
fig2.show()

print("\n💡 Observación:")
print("  - La política óptima tiene valores significativamente más altos")
print("  - El gradiente de valores apunta hacia el objetivo")
print("  - Las ecuaciones de Bellman permiten calcular estos valores eficientemente")

## 🔧 Versión con Framework

Ahora usaremos **Gymnasium** con el ambiente FrozenLake y aplicaremos los mismos algoritmos.

In [ ]:
import gymnasium as gym

# Crear ambiente
env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=False)

print("🧊 FrozenLake-v1 Environment\n")
print(f"  Estados: {env.observation_space.n}")
print(f"  Acciones: {env.action_space.n}")
print(f"  Ambiente: 4x4 grid\n")

# Implementar Value Iteration para FrozenLake
def value_iteration_gym(env, gamma=0.99, theta=1e-8, max_iterations=1000):
    """
    Value Iteration para ambientes de Gymnasium.
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    
    V = np.zeros(n_states)
    
    for iteration in range(max_iterations):
        delta = 0
        V_old = V.copy()
        
        for s in range(n_states):
            action_values = []
            
            for a in range(n_actions):
                # Simular la acción (Gymnasium no expone P directamente en algunos ambientes)
                # Para FrozenLake necesitamos acceder a la dinámica interna
                q_value = 0
                
                # FrozenLake tiene acceso a P
                if hasattr(env.unwrapped, 'P'):
                    for prob, next_state, reward, done in env.unwrapped.P[s][a]:
                        q_value += prob * (reward + gamma * V_old[next_state])
                
                action_values.append(q_value)
            
            V[s] = max(action_values) if action_values else 0
            delta = max(delta, abs(V[s] - V_old[s]))
        
        if delta < theta:
            print(f"✅ Convergió en {iteration + 1} iteraciones")
            break
    
    # Extraer política
    policy = np.zeros(n_states, dtype=int)
    for s in range(n_states):
        action_values = []
        for a in range(n_actions):
            q_value = 0
            if hasattr(env.unwrapped, 'P'):
                for prob, next_state, reward, done in env.unwrapped.P[s][a]:
                    q_value += prob * (reward + gamma * V[next_state])
            action_values.append(q_value)
        policy[s] = np.argmax(action_values)
    
    return V, policy

print("🎯 Ejecutando Value Iteration en FrozenLake...\n")
V_fl, policy_fl = value_iteration_gym(env, gamma=0.99)

print("\nFunción de valor óptima:")
print(V_fl.reshape(4, 4).round(3))

print("\nPolítica óptima:")
action_symbols = ['←', '↓', '→', '↑']
policy_grid = policy_fl.reshape(4, 4)
for row in policy_grid:
    print(' '.join([action_symbols[a] for a in row]))

env.close()

## 🎯 Ejercicios

### 🟢 Ejercicio 1: Impacto del Factor de Descuento

Experimenta con diferentes valores de γ y observa cómo afecta la política.

In [ ]:
def ejercicio_1_discount_factor():
    """
    Objetivo: Comprender el efecto del factor de descuento.
    
    Instrucciones:
    1. Ejecuta value_iteration con γ = 0.5, 0.9, 0.99
    2. Compara las funciones de valor resultantes
    3. Retorna un diccionario con los valores máximos para cada γ
    """
    # TODO: Tu código aquí
    pass

def test_ejercicio_1():
    resultado = ejercicio_1_discount_factor()
    assert resultado is not None, "❌ Pista: Debes retornar un diccionario"
    assert isinstance(resultado, dict), "❌ Pista: El resultado debe ser un diccionario"
    print("✅ ¡Correcto!")
    print("   γ más bajo → valores más bajos (el agente es miope)")
    print("   γ más alto → valores más altos (el agente es previsor)")
    return True

# test_ejercicio_1()  # Descomenta para probar

### 🟡 Ejercicio 2: Policy Iteration

Implementa el algoritmo de Policy Iteration, que alterna entre evaluación y mejora de políticas.

In [ ]:
def policy_iteration(mdp: GridWorldMDP, max_iterations: int = 100) -> Tuple[np.ndarray, np.ndarray]:
    """
    Implementa Policy Iteration.
    
    Algoritmo:
    1. Inicializar política aleatoria
    2. Loop:
       a. Policy Evaluation: calcular V^π
       b. Policy Improvement: mejorar π usando V^π
       c. Si π no cambió, terminar
    
    Instrucciones:
    - Usa policy_evaluation() para evaluar
    - Para mejorar, elige acción greedy respecto a V
    - Retorna V y política final
    """
    # TODO: Tu código aquí
    # Hint: Inicializa con política uniforme
    # Hint: Para mejorar, usa argmax sobre Q(s,a)
    pass

def test_ejercicio_2():
    mdp_test = GridWorldMDP(size=4, gamma=0.9)
    V, policy = policy_iteration(mdp_test)
    
    assert V is not None, "❌ Pista: Debes retornar V y policy"
    assert V.shape == (4, 4), "❌ Pista: V debe tener shape (4, 4)"
    print("✅ ¡Excelente! Policy Iteration implementado correctamente.")
    print("   Policy Iteration converge en menos iteraciones que Value Iteration.")
    return True

# test_ejercicio_2()  # Descomenta para probar

### 🔴 Ejercicio 3: MDP Estocástico

Modifica el MDP para que las acciones sean estocásticas (el agente se mueve en la dirección deseada con probabilidad 0.8).

In [ ]:
class StochasticGridWorldMDP(GridWorldMDP):
    """
    GridWorld con transiciones estocásticas.
    
    Con probabilidad 0.8, el agente se mueve en la dirección deseada.
    Con probabilidad 0.1, se mueve perpendicular (izquierda o derecha).
    
    Objetivo: Implementar get_transition_prob() para reflejar esto.
    """
    
    def __init__(self, size: int = 4, gamma: float = 0.9, slip_prob: float = 0.2):
        super().__init__(size, gamma)
        self.slip_prob = slip_prob
    
    def get_transition_prob(self, state: Tuple[int, int], action: int, next_state: Tuple[int, int]) -> float:
        """
        TODO: Implementa transiciones estocásticas.
        
        Hint:
        - Probabilidad principal (1 - slip_prob) a la acción deseada
        - Probabilidad slip_prob/2 a cada acción perpendicular
        - Calcula los 3 posibles next_states
        - Retorna la probabilidad correspondiente
        """
        # TODO: Tu código aquí
        pass

def test_ejercicio_3():
    stochastic_mdp = StochasticGridWorldMDP(size=4, gamma=0.9, slip_prob=0.2)
    
    # Probar que las probabilidades suman 1
    state = (1, 1)
    action = 0  # arriba
    total_prob = sum(stochastic_mdp.get_transition_prob(state, action, s) for s in stochastic_mdp.states)
    
    assert abs(total_prob - 1.0) < 1e-6, "❌ Pista: Las probabilidades deben sumar 1"
    print("✅ ¡Excelente! MDP estocástico implementado correctamente.")
    print("   Los MDPs estocásticos modelan incertidumbre en el ambiente.")
    return True

# test_ejercicio_3()  # Descomenta para probar

## 📚 Resumen

### Conceptos Clave

- **MDP (Markov Decision Process)**: Formalización matemática completa de problemas de RL
- **Propiedad de Markov**: El futuro depende solo del presente, no del pasado
- **Función de Valor V(s)**: Retorno esperado comenzando en estado s
- **Función Q(s,a)**: Retorno esperado tomando acción a en estado s
- **Factor de Descuento γ**: Controla importancia de recompensas futuras
- **Ecuaciones de Bellman**: Relaciones recursivas entre valores de estados
- **Policy Evaluation**: Calcular V^π dada una política π
- **Value Iteration**: Encontrar V* directamente usando optimalidad de Bellman
- **Policy Iteration**: Alternar entre evaluación y mejora de política

### Ecuaciones Fundamentales

| Ecuación | Fórmula | Uso |
|----------|---------|-----|
| **Bellman (V)** | $V^{\pi}(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)[R + \gamma V^{\pi}(s')]$ | Evaluar política |
| **Bellman (Q)** | $Q^{\pi}(s,a) = \sum_{s'} P(s'|s,a)[R + \gamma \sum_{a'} \pi(a'|s') Q^{\pi}(s',a')]$ | Evaluar acciones |
| **Optimalidad (V)** | $V^*(s) = \max_a \sum_{s'} P(s'|s,a)[R + \gamma V^*(s')]$ | Encontrar política óptima |
| **Optimalidad (Q)** | $Q^*(s,a) = \sum_{s'} P(s'|s,a)[R + \gamma \max_{a'} Q^*(s',a')]$ | Q-Learning (siguiente notebook) |

### Algoritmos

| Algoritmo | Complejidad | Ventajas | Desventajas |
|-----------|-------------|----------|-------------|
| **Policy Evaluation** | O(\|S\|² \|A\|) por iteración | Simple, garantiza convergencia | Solo evalúa, no optimiza |
| **Value Iteration** | O(\|S\|² \|A\|) por iteración | Encuentra óptimo directamente | Puede ser lento |
| **Policy Iteration** | O(\|S\|² \|A\|) por iteración | Converge más rápido | Más complejo |

### Lo que viene

Los métodos vistos requieren:
- Conocer la dinámica del MDP (P, R)
- Espacio de estados pequeño (tabular)

En el siguiente notebook veremos **Q-Learning**, que:
- Aprende sin conocer P y R (model-free)
- Usa muestreo en lugar de cálculo exhaustivo
- Es la base de Deep RL moderno

## 🔗 Recursos Adicionales

### 📄 Papers Fundamentales

- **"A Markovian Decision Process"** - Bellman (1957)
  - Paper original que introduce MDPs
  - Contexto histórico: Base matemática de RL

- **"Dynamic Programming and Markov Processes"** - Howard (1960)
  - Desarrolla algoritmos de programación dinámica
  - Aplicaciones prácticas de MDPs

### 📖 Capítulos de Libros

- **Sutton & Barto - Capítulos 3-4**
  - Capítulo 3: Finite MDPs
  - Capítulo 4: Dynamic Programming
  - Explicaciones detalladas con ejemplos

### 🎥 Videos Recomendados

- **David Silver - Lecture 2: Markov Decision Processes**
  - https://www.youtube.com/watch?v=lfHX2hHRMVQ
  - Explicación clara de MDPs y ecuaciones de Bellman

- **David Silver - Lecture 3: Planning by Dynamic Programming**
  - https://www.youtube.com/watch?v=Nd1-UUMVfz4
  - Algoritmos de programación dinámica

### 💻 Implementaciones

- **GridWorld Implementations**
  - https://github.com/dennybritz/reinforcement-learning
  - Implementaciones limpias de DP algorithms

## ➡️ Próximo Paso

En el siguiente notebook aprenderás sobre **Q-Learning**, el primer algoritmo model-free que aprende directamente de la experiencia sin necesitar conocer la dinámica del ambiente.

**[Continuar con: 03. Q-Learning →](03-q-learning.ipynb)**

---

<div align="center">
    
**¡Has dominado los fundamentos matemáticos del RL! 🎉**

</div>